go mod is Go’s dependency + version management system.

Before modules → Go was painful.
Now → Every project is a self-contained module.

In [ ]:
go mod init github.com/yourname/go-backend-mastery
//go: creating new go.mod: module github.com/yourname/go-backend-mastery


📦 What Happens When You Add Dependencies?

If you install something like:

go get github.com/gin-gonic/gin


Go updates:

go.mod

go.sum

go.sum = checksum for security.

Never delete these manually.

Project Structure 
go-backend-mastery/
│
├── go.mod
├── main.go
│
├── internal/
│   └── user/
│       └── service.go
│
└── pkg/
    └── utils/
        └── helper.go


🟢 internal/

Code that should NOT be imported by other projects.

Go enforces this.

Anything inside internal/:

Only usable within this module

Not importable externally

Used in production systems heavily.

🟢 pkg/

Reusable code.

Other projects can import this if your module is public.

Part 3 — Packages

In Go:

Every folder is a package.

Not every file. Folder = package.

🔹 Example

Create:

internal/user/service.go


Add:

package user

func GetUserName() string {
    return "Rishav"
}


Now in main.go:

package main

import (
    "fmt"
    "github.com/yourname/go-backend-mastery/internal/user"
)

func main() {
    fmt.Println(user.GetUserName())
}


Run:

go run .

🧠 Key Rule

The package name must match folder purpose, not folder name strictly — but conventionally they match.

❗ Important

All files in same folder must have same package name.

You cannot mix:

package user
package utils


Inside same folder.

Part 4 — Visibility Rules (Very Important)

Go has no public, private, protected.

It uses:

Capitalization.

🔓 Exported (Public)

Starts with CAPITAL letter.

func GetUser() {}
type User struct {}


These can be accessed outside package.

🔒 Unexported (Private)

Starts with lowercase.

func getUser() {}
type user struct {}


Only accessible within same package.

🔎 Example

Inside internal/user/service.go:

package user

type User struct {
    Name string
    age  int
}

func NewUser(name string, age int) User {
    return User{
        Name: name,
        age:  age,
    }
}

func (u User) GetAge() int {
    return u.age
}


From main.go:

u := user.NewUser("Rishav", 25)

fmt.Println(u.Name)     // works
fmt.Println(u.GetAge()) // works

// fmt.Println(u.age)   ❌ won't work (private field)

🧠 Important Design Philosophy

Go encourages:

Small packages

Clear exports

Minimal public API

Unlike Java (everything public by default in enterprise projects), Go forces you to think about boundaries.

# The Go Model: The "M:N" Scheduler
Go uses a user-space scheduler that sits between your code and the OS. It employs the G-M-P model to distribute work across a multi-threaded CPU.
+1

G (Goroutine): A lightweight "virtual" thread (starts at ~2KB).

M (Machine): An actual OS Thread managed by the kernel.

P (Processor): A logical resource/context required to execute Go code (usually matches the number of CPU cores).

How it works on a Multi-threaded CPU:
The Go runtime creates a pool of OS Threads (M). If you have a 16-core CPU, Go will typically set up 16 P contexts. Each P has a local queue of Gs waiting to run.
+1

Work Stealing: If one CPU core finishes its tasks and its queue is empty, the Go scheduler will "steal" half of the Goroutines from another core's queue. This ensures all your CPU cores stay at 100% utilization.

Syscall Handling: When a Goroutine makes a blocking system call (like reading a huge file), Go is smart. It detaches the OS thread (M) from the CPU context (P) so the CPU can keep running other Goroutines on a fresh thread while the first one waits for the disk.

# The Rust Model: "Fearless Concurrency"

Unlike Go, Rust does not have a built-in "global" scheduler. Instead, it gives you two distinct paths to use your multi-threaded CPU:Path A: OS Threads (std::thread)Rust's standard library uses 1:1 Threading. When you call thread::spawn, you are asking the OS for a real, heavy-weight thread.Best for: CPU-heavy math or data processing.CPU Performance: The OS kernel handles the scheduling. Rust ensures this is safe via Ownership. You cannot have two threads accidentally changing the same piece of memory at once because the compiler will stop you.+1Path B: Async/Await (The "Tokio" Model)For high-speed web servers, Rust uses Asynchronous Tasks (similar to Goroutines) usually managed by a library like Tokio.Task-Stealing: Similar to Go, Tokio runs a "multi-threaded runtime." It spawns one OS thread per CPU core and uses a "work-stealing" scheduler to move tasks between cores.+1Zero-Cost: Because Rust has no Garbage Collector, the CPU spends 0% of its time "cleaning up" memory during execution. Every single clock cycle goes toward your actual logic.

# Comparison Summary: Multi-core Execution

Feature                     Go (Goroutines)                       Rust (Tokio/Async)Scheduling                 Automatic, built-in to the language.  Explicit; you choose a runtime (like Tokio).
CPU Efficiency             High, but Garbage Collector (GC) occasionally pauses threads.Maximum; no GC pauses, code runs at "native" speed.

Memory Safety             Handled at runtime (GC).               Handled at compile-time (Borrow Checker).

Context Switching         Very fast (managed in user-space).     Very fast (managed in user-space).